# motion-kit: video plates on free Colab (Wan 2.2 5B)

Makes the **video plates** of a production's job file with **Wan 2.2 TI2V-5B** (Apache-2.0) on a free T4.

1. On your Mac: `npm run produce -- specs/productions/<name>.json --tier local` → writes `motion-kit/out/<name>.jobs.json`.
2. Here: **Runtime → Change runtime type → T4 GPU**, then run all cells and upload that `jobs.json`.
3. Download the zip, unzip it into `motion-kit/public/` (it contains `plates/<name>/<id>.mp4`), run `produce` again.

Expect roughly 15-40 minutes per 5 s clip at 480p on a T4 (estimate; the model card asks for 24 GB, so this uses CPU offload).
Not run by the author yet: the first run is the test. If it runs out of memory, lower `FRAMES` or use a ZeroGPU Space
(e.g. `Wan-AI/Wan-2.2-5B`, 5 free GPU-minutes a day) or the Draw Things app on the Mac.


In [ ]:
!pip -q install git+https://github.com/huggingface/diffusers ftfy accelerate imageio-ffmpeg transformers sentencepiece

In [ ]:
import json
from google.colab import files
up = files.upload()                      # pick out/<name>.jobs.json
doc = json.loads(next(iter(up.values())))
jobs = [j for j in doc["jobs"] if j["type"] == "video"]
print(doc["video"], "·", len(jobs), "video job(s)")
for j in jobs: print("-", j["id"], j.get("seconds"), "s", j.get("aspect"), "|", j["prompt"][:100], "…")

In [ ]:
import torch
from diffusers import WanPipeline, AutoencoderKLWan
from diffusers.utils import export_to_video
MID = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
vae = AutoencoderKLWan.from_pretrained(MID, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MID, vae=vae, torch_dtype=torch.float16)  # T4 has no fast bf16
pipe.enable_model_cpu_offload()                                               # fits 16 GB by streaming weights

In [ ]:
import os, math
SIZES = {"9:16": (832, 480), "16:9": (480, 832), "1:1": (640, 640), "4:5": (800, 640)}   # (height, width), 480p class
FPS, STEPS, FRAMES = 24, 30, None   # FRAMES=None: from each job's seconds (4k+1 frames, max 121)
NEG = "text, subtitles, logo, watermark, blurry, distorted face, extra fingers, low quality"
os.makedirs("out", exist_ok=True)
for j in jobs:
    h, w = SIZES.get(j.get("aspect", "9:16"), (832, 480))
    n = FRAMES or min(121, int(j.get("seconds", 5) * FPS) // 4 * 4 + 1)
    print(f"{j['id']}: {w}x{h}, {n} frames …")
    frames = pipe(prompt=j["prompt"], negative_prompt=NEG, height=h, width=w, num_frames=n,
                  guidance_scale=5.0, num_inference_steps=STEPS).frames[0]
    path = os.path.join("out", j["out"])
    os.makedirs(os.path.dirname(path), exist_ok=True)
    export_to_video(frames, path, fps=FPS)
    print("  saved", path)

In [ ]:
import shutil
zipname = shutil.make_archive(f"{doc['video']}-plates", "zip", "out")
files.download(zipname)   # unzip into motion-kit/public/, then: npm run produce -- … --tier local